# 12 — Persona × CLV-proxy

Giữ cố định sáu persona cây. Dùng output 07 có sẵn, không train hay chọn lại mô hình từ kết quả mới. CLV-proxy ở đây là **điểm acquisition dự kiến trong H tháng**, không phải tiền, lợi nhuận hay CLV trọn đời; predicted_points không phải xác suất mua.

**Đầu vào:** output 11 (persona_lifecycle_2016-05.parquet, tất cả known customers), output 10 (personas_tree_YYYY-MM.parquet, để backtest), output 07 (clv_predictions, test_predictions, selection.json). Thêm ba nguồn vào Kaggle và sửa folder nếu có nhiều bản. Cần pandas/numpy/pyarrow/matplotlib.

Mặc định HORIZON=1, TARGET='points', cùng nhiệm vụ acquisition tháng sau hiện tại. Có thể chạy riêng H=3 nếu đủ files; không trộn score hay metric của hai horizon. Các tier chỉ là ưu tiên trong từng persona, không tạo persona mới. Affinity sẽ bổ sung sau.

Câu hỏi: persona nào có CLV-proxy cao? Trong cùng persona, điểm có giúp chọn khách tốt hơn chọn ngẫu nhiên không? New và khách ngoài cây có dự báo không?

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
PERSONA_LIFECYCLE_ROOT=None
TREE_ROOT=None
CLV_ROOT=None
RECOMMEND_MONTH='2016-05'
HORIZON=1
TARGET='points'
EVAL_MONTHS=['2015-12','2016-01','2016-02','2016-03','2016-04']
MIN_N=1000;MIN_ACTUAL_POSITIVE=20
OUT=Path('/kaggle/working/persona_clv_outputs' if Path('/kaggle/working').exists() else './persona_clv_outputs')
OUT.mkdir(parents=True,exist_ok=True)
SEARCH=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
def resolve(configured,pattern):
    root=Path(configured) if configured else SEARCH
    candidates=sorted({p.parent for p in root.rglob(pattern)})
    if len(candidates)!=1:raise ValueError(f'{pattern}: đặt folder duy nhất trong {candidates}')
    return candidates[0]
P=resolve(PERSONA_LIFECYCLE_ROOT,f'persona_lifecycle_{RECOMMEND_MONTH}.parquet')
T=resolve(TREE_ROOT,'personas_tree_????-??.parquet')
C=resolve(CLV_ROOT,f'clv_predictions_{RECOMMEND_MONTH}_{HORIZON}m.parquet')
selection=json.loads((C/'selection.json').read_text())
assert str(HORIZON) in selection['selected_by_horizon'],'Chỉ dùng horizon có selection validation đã khóa'
METHOD=selection['selected_by_horizon'][str(HORIZON)][TARGET]
SPEC=selection['backtest_specs'][str(HORIZON)]
assert TARGET in ['points','acquisitions','event_months']
assert HORIZON in [1,3]
def unique(d):assert d.customer_id.notna().all() and d.customer_id.is_unique

def validate_numeric(d,col):
    assert pd.to_numeric(d[col],errors='coerce').notna().all(),f'{col}: missing/non-numeric'
    assert np.isfinite(d[col].to_numpy(float)).all() and d[col].ge(0).all(),f'{col}: invalid'

def latest_join():
    known=pd.read_parquet(P/f'persona_lifecycle_{RECOMMEND_MONTH}.parquet');unique(known)
    assert known.as_of_month.astype(str).eq(RECOMMEND_MONTH).all()
    pred=pd.read_parquet(C/f'clv_predictions_{RECOMMEND_MONTH}_{HORIZON}m.parquet');unique(pred)
    assert pred.as_of_month.astype(str).eq(RECOMMEND_MONTH).all() and pred.horizon_months.eq(HORIZON).all()
    assert pred['method_'+TARGET].eq(METHOD).all(),'Current model không khớp selection'
    validate_numeric(pred,'predicted_'+TARGET)
    assert set(pred.customer_id)<=set(known.customer_id),'07/11 khác population'
    if 'lifecycle' in pred:
        audit=pred[['customer_id','lifecycle']].merge(known[['customer_id','lifecycle']],on='customer_id',validate='one_to_one',suffixes=('_07','_11'))
        assert audit.lifecycle_07.eq(audit.lifecycle_11).all(),'07/11 lifecycle khác phiên bản'
    cols=['customer_id','predicted_'+TARGET,'method_'+TARGET]
    d=known.merge(pred[cols],on='customer_id',how='left',validate='one_to_one')
    d['clv_score']=d['predicted_'+TARGET]
    d['has_clv']=d.clv_score.notna()
    d['persona_group']=d.persona_id.astype('string').fillna('NoCurrentPersona')
    d['clv_horizon_months']=HORIZON;d['clv_target']=TARGET
    return d
current=latest_join()
print('CLV model khóa từ validation:',METHOD,'horizon:',HORIZON,'target:',TARGET)

## 1. Coverage và phân bố dự báo
Coverage mẫu số là toàn known population của 11, không chỉ khách được dự báo. Không gán CLV=0 cho khách thiếu dự báo. NoCurrentPersona gồm nhiều trạng thái khác nhau, phải đọc kèm lifecycle; New có thể có CLV dù chưa có persona cây.

Báo trung vị, p25/p75/p90 và trung bình. Trung bình có thể bị đuôi score chi phối. Chênh score dự báo chưa chứng minh chênh outcome thật.

In [ ]:
coverage=current.groupby(['persona_group','lifecycle'],observed=True).agg(known_n=('customer_id','size'),predicted_n=('has_clv','sum')).reset_index()
coverage['clv_coverage']=coverage.predicted_n/coverage.known_n
coverage.to_csv(OUT/'clv_coverage_persona_lifecycle.csv',index=False);display(coverage)
scored=current.loc[current.has_clv].copy()
summary=scored.groupby(['persona_group','lifecycle'],observed=True).clv_score.agg(n='size',mean='mean',median='median',p25=lambda s:s.quantile(.25),p75=lambda s:s.quantile(.75),p90=lambda s:s.quantile(.9),max='max').reset_index()
summary.to_csv(OUT/'clv_distribution_persona_lifecycle.csv',index=False);display(summary)
by_persona=scored.groupby('persona_group').clv_score.agg(n='size',mean='mean',median='median',p90=lambda s:s.quantile(.9)).reset_index()
by_persona.to_csv(OUT/'clv_distribution_persona.csv',index=False)
fig,ax=plt.subplots(figsize=(10,5))
x=np.arange(len(by_persona));ax.bar(x-.18,by_persona['mean'],width=.36,label='Mean');ax.bar(x+.18,by_persona['median'],width=.36,label='Median')
ax.set_xticks(x,by_persona.persona_group,rotation=25);ax.set(ylabel=f'Dự báo {TARGET} / {HORIZON} tháng',title='CLV-proxy theo persona — chưa phải outcome thực tế');ax.legend()
plt.tight_layout();plt.savefig(OUT/'clv_distribution.png',bbox_inches='tight');plt.show()


## 2. Ưu tiên trong persona hiện tại
Tier học từ **phân bố dự báo hiện tại** để mô tả/triển khai, không dùng nhãn tương lai. Lưu biên để áp dụng lần sau; không dùng các biên May để đánh giá tháng quá khứ. Điểm bằng nhau giữ cùng tier. Nếu mọi điểm bằng nhau, AllEqual, không giả tạo bốn mức.

Tier của hai persona không tương đương về score: T4 trong P1 có thể thấp hơn T1 trong P4. Để chọn khách trên toàn population, đọc clv_score cùng horizon/target/model. NoCurrentPersona chia riêng theo lifecycle để tránh trộn New với nhóm khác. Thiếu CLV giữ MissingPrediction.

In [ ]:
current['priority_context']=np.where(current.persona_group.eq('NoCurrentPersona'),'NoCurrentPersona | '+current.lifecycle.astype(str),current.persona_group.astype(str))
current['clv_tier']='MissingPrediction';tier_specs={}
for context,ix in current.loc[current.has_clv].groupby('priority_context').groups.items():
    s=current.loc[ix,'clv_score']
    if s.nunique()==1:
        current.loc[ix,'clv_tier']='AllEqual';tier_specs[context]={'all_equal':True,'score':float(s.iloc[0])};continue
    boundaries=np.unique(s.quantile([.25,.5,.75]).to_numpy(float))
    edges=np.r_[-np.inf,boundaries,np.inf]
    codes=pd.cut(s,edges,labels=False,include_lowest=True)
    current.loc[ix,'clv_tier']=['T'+str(int(i)+1) for i in codes]
    tier_specs[context]={'all_equal':False,'internal_boundaries':boundaries.tolist(),'boundary_inclusion':'right; score<=boundary in lower tier'}
current.to_parquet(OUT/f'persona_clv_{RECOMMEND_MONTH}_{HORIZON}m.parquet',index=False)
current.loc[current.has_clv].to_parquet(OUT/f'priority_candidates_{RECOMMEND_MONTH}_{HORIZON}m.parquet',index=False)
priority=current.loc[current.has_clv].groupby(['priority_context','clv_tier']).clv_score.agg(n='size',min='min',median='median',max='max').reset_index()
priority.to_csv(OUT/'current_priority_tiers.csv',index=False);display(priority)
(OUT/'priority_tier_specs.json').write_text(json.dumps({'as_of_month':RECOMMEND_MONTH,'horizon':HORIZON,'target':TARGET,'specs':tier_specs},ensure_ascii=False,indent=2),encoding='utf-8')


## 3. Backtest: CLV có thêm ích lợi trong cùng persona không?
Dùng test_predictions của 07, đúng method khóa bằng validation; không dùng forecast May để chấm quá khứ. Cùng persona mà chỉ chọn ngẫu nhiên thì capture kỳ vọng = k/n và lift=1. So CLV với baseline này bằng top10% **trong từng nhóm**.

Target points là tổng điểm acquisition thực tế, không phải binary conversion. Capture là phần tổng điểm được chọn; Lift10 là điểm thực tế bình quân top10% / bình quân nhóm. Ties dùng fractional selection. Điểm hằng vẫn báo kết quả ngẫu nhiên và ranking_informative=False; nếu total_actual=0 thì capture/lift/calibration để NaN.

Backtest 07 chỉ chứa khách đủ nhãn tương lai. Mẫu số prediction_join_coverage là cohort có nhãn này, không phải toàn known population. Dùng bảng coverage hiện tại ở phần 1 để hiểu khách bị loại. NoCurrentPersona backtest chia theo lifecycle. Không dùng tiny-group lift làm kết luận.

In [ ]:
def score_metrics(y,p):
    y=np.asarray(y,float);p=np.asarray(p,float);n=len(y)
    assert n and np.isfinite(y).all() and np.isfinite(p).all() and (y>=0).all() and (p>=0).all()
    k=max(1,int(np.ceil(.1*n)));threshold=np.partition(p,n-k)[n-k]
    high=p>threshold;ties=p==threshold
    selected=y[high].sum()+(k-int(high.sum()))*y[ties].mean()
    total=y.sum();positive=int((y>0).sum())
    return dict(n=n,actual_positive_n=positive,actual_mean=y.mean(),predicted_mean=p.mean(),
        MAE=np.abs(y-p).mean(),RMSE=np.sqrt(((y-p)**2).mean()),
        calibration_ratio=p.sum()/total if total else np.nan,
        capture10=selected/total if total else np.nan,
        lift10=(selected/k)/(total/n) if total else np.nan,
        random_capture=k/n,random_lift=1. if total else np.nan,
        ranking_informative=bool(np.unique(p).size>1),weak_evidence=n<MIN_N or positive<MIN_ACTUAL_POSITIVE)
metrics=[];coverage_test=[];status=[]
for m in EVAL_MONTHS:
    if m not in SPEC['test']:
        status.append(dict(month=m,status='not_in_locked_test_spec'));continue
    assert pd.Period(SPEC['validation'],'M')+HORIZON<=pd.Period(m,'M'),'Nhãn validation chưa có tại cutoff'
    path=C/f'test_predictions_{HORIZON}m_{m}_{TARGET}.parquet'
    if not path.exists():
        status.append(dict(month=m,status='missing_test_prediction'));continue
    d=pd.read_parquet(path);unique(d)
    assert d.month.astype(str).eq(m).all() and d.horizon.eq(HORIZON).all() and d.target.eq(TARGET).all()
    assert d.method.eq(METHOD).all(),'Backtest method khác selection'
    validate_numeric(d,'actual');validate_numeric(d,'prediction')
    p=pd.read_parquet(T/f'personas_tree_{m}.parquet');unique(p)
    assert p.as_of_month.astype(str).eq(m).all()
    # 07 lifecycle nguồn test, 10 chỉ bổ sung persona; audit tag nếu có.
    if 'lifecycle' in p:
        a=d[['customer_id','lifecycle']].merge(p[['customer_id','lifecycle']],on='customer_id',how='inner',validate='one_to_one',suffixes=('_07','_10'))
        assert a.lifecycle_07.eq(a.lifecycle_10).all(),'Lifecycle lịch sử khác phiên bản'
    d=d.merge(p[['customer_id','persona_id']],on='customer_id',how='left',validate='one_to_one')
    d['persona_group']=d.persona_id.astype('string').fillna('NoCurrentPersona')
    d['evaluation_group']=np.where(d.persona_id.notna(),d.persona_group,'NoCurrentPersona | '+d.lifecycle.astype(str))
    coverage_test.append(dict(month=m,labelled_prediction_n=len(d),joined_persona_n=int(d.persona_id.notna().sum()),persona_join_coverage=d.persona_id.notna().mean()))
    for group,g in d.groupby('evaluation_group'):
        metrics.append(dict(month=m,group=group,**score_metrics(g.actual,g.prediction)))
    metrics.append(dict(month=m,group='ALL_LABELLED_PREDICTIONS',**score_metrics(d.actual,d.prediction)))
    status.append(dict(month=m,status='evaluated'))
status=pd.DataFrame(status);status.to_csv(OUT/'backtest_status.csv',index=False);display(status)
pd.DataFrame(coverage_test).to_csv(OUT/'prediction_join_coverage.csv',index=False)
metrics=pd.DataFrame(metrics)
if len(metrics):
    metrics.to_csv(OUT/'clv_within_persona_backtest.csv',index=False);display(metrics.round(4))
    review=metrics.groupby('group').agg(months=('month','nunique'),lift10_min=('lift10','min'),lift10_mean=('lift10','mean'),capture10_mean=('capture10','mean'),calibration_min=('calibration_ratio','min'),calibration_max=('calibration_ratio','max'),strong_months=('weak_evidence',lambda s:int((~s).sum())),informative_months=('ranking_informative','sum')).reset_index()
    review.to_csv(OUT/'clv_within_persona_summary.csv',index=False);display(review.round(4))
else:print('CHƯA có backtest: xem backtest_status.csv; không kết luận CLV giúp chọn khách.')


## 4. Thẻ CLV và bước đọc kết quả
Giữ 6 persona; CLV bổ sung mức ưu tiên. Chỉ kết luận CLV thêm ích lợi khi lift trong persona bền qua tháng, đủ sample, score có phân biệt. Calibration_ratio xa 1: có thể xếp hạng hữu ích nhưng dự báo lượng sai mức; không dùng làm forecast ngân sách/tiền.

New có thể được dự báo dù chưa đủ lịch sử cây: giữ nhánh NoCurrentPersona | New riêng. At-risk/Churned thiếu snapshot không được tự điền CLV=0.

Không đánh giá affordability, khả năng đáp ứng điều kiện sản phẩm hay hiệu quả can thiệp. Bước affinity sẽ chọn sản phẩm chưa sở hữu; CLV không chỉ định sản phẩm, không nhân score với affinity rồi gọi là doanh thu.

In [ ]:
cards=[]
for group,g in current.groupby('persona_group'):
    predicted=g.loc[g.has_clv,'clv_score']
    desc=f'median={predicted.median():.4f}, p90={predicted.quantile(.9):.4f}' if len(predicted) else 'Chưa có dự báo'
    cards.append(f'## {group}\n- Known n={len(g):,}; có CLV={len(predicted):,} ({len(predicted)/len(g):.1%}).\n- CLV {TARGET} / {HORIZON} tháng: {desc}.\n- Đây là dự báo, không phải tiền hay outcome tháng tới đã quan sát.\n')
(OUT/'persona_clv_cards.md').write_text('# Persona × CLV-proxy — '+RECOMMEND_MONTH+'\n\n'+'\n'.join(cards),encoding='utf-8')
(OUT/'config.json').write_text(json.dumps(dict(persona_lifecycle_root=str(P),tree_root=str(T),clv_root=str(C),recommend_month=RECOMMEND_MONTH,horizon=HORIZON,target=TARGET,selected_method=METHOD,eval_months=EVAL_MONTHS,scope='development backtest; priority within persona; no model refit or reselection; points not money'),ensure_ascii=False,indent=2),encoding='utf-8')
print('Outputs:',OUT)


## 5. Bổ sung — lift tổng và khoảng acquisition

Dùng **cùng cohort tất cả labelled predictions của 07 tại cùng tháng, horizon, target** làm baseline. Đây không phải baseline toàn bộ known customers: khách thiếu nhãn tương lai không có trong cohort này.

- Lift persona = mean actual nhóm / mean actual toàn labelled cohort.
- Lift10 nội nhóm = mean actual top10% nhóm / mean actual nhóm.
- Lift tổng top10% nhóm = mean actual top10% nhóm / mean actual toàn labelled cohort.
- Tích hai lift bằng lift tổng **ở từng tháng**; không nhân hai trung bình qua tháng.

Tính riêng **điểm acquisition** (`actual`) và **tỷ lệ có acquisition** (`actual > 0`). Với weights dương, actual>0 nhận biết có event; không diễn giải points thành xác suất hay tiền.

Báo hai loại range: (1) min–max của trung bình thực tế từng tháng, chỉ là biến động quá khứ, không phải prediction interval; (2) P10–P90 score dự báo hiện tại, là phân bố giữa khách, không phải CI. Thêm actual P10/P90 và P10/P90 trong khách có actual>0; đa số actual có thể bằng 0 nên quantile toàn nhóm dễ cùng bằng 0.

Top10% xếp bằng CLV score đã có, cùng fractional tie policy. Không đổi model hoặc persona. NoCurrentPersona tiếp tục chia riêng theo lifecycle. Kết quả là development backtest.

In [ ]:
def top_mean_with_ties(y,score):
    y=np.asarray(y,float);score=np.asarray(score,float)
    assert len(y)>0 and len(y)==len(score) and np.isfinite(y).all() and np.isfinite(score).all()
    k=max(1,int(np.ceil(.1*len(y))))
    threshold=np.partition(score,len(score)-k)[len(score)-k]
    above=score>threshold;ties=score==threshold
    selected=y[above].sum()+(k-int(above.sum()))*y[ties].mean()
    return float(selected/k),k

lift_rows=[];lift_status=[]
for m in EVAL_MONTHS:
    if m not in SPEC['test']:
        lift_status.append(dict(month=m,status='not_in_locked_test_spec'));continue
    f=C/f'test_predictions_{HORIZON}m_{m}_{TARGET}.parquet'
    if not f.exists():
        lift_status.append(dict(month=m,status='missing_test_prediction'));continue
    d=pd.read_parquet(f);unique(d)
    assert d.month.astype(str).eq(m).all() and d.horizon.eq(HORIZON).all() and d.target.eq(TARGET).all()
    assert d.method.eq(METHOD).all()
    validate_numeric(d,'actual');validate_numeric(d,'prediction')
    p=pd.read_parquet(T/f'personas_tree_{m}.parquet');unique(p)
    assert p.as_of_month.astype(str).eq(m).all()
    d=d.merge(p[['customer_id','persona_id']],on='customer_id',how='left',validate='one_to_one')
    d['evaluation_group']=np.where(d.persona_id.notna(),d.persona_id.astype('string').fillna('NoCurrentPersona'),
        'NoCurrentPersona | '+d.lifecycle.astype(str))
    global_mean=float(d.actual.mean());global_rate=float(d.actual.gt(0).mean())
    # ALL is an aggregate row; never sum it together with group rows.
    groups=list(d.groupby('evaluation_group'))+[('ALL_LABELLED_PREDICTIONS',d)]
    for group,g in groups:
        y=g.actual.to_numpy(float);score=g.prediction.to_numpy(float);yb=(y>0).astype(float)
        mean=float(y.mean());rate=float(yb.mean())
        top_mean,k=top_mean_with_ties(y,score);top_rate,_=top_mean_with_ties(yb,score)
        positive=y[y>0]
        row=dict(month=m,group=group,n=len(g),global_labelled_n=len(d),top_selected_n=k,
            actual_positive_n=len(positive),actual_mean=mean,global_actual_mean=global_mean,
            top10_actual_mean=top_mean,
            lift_persona=mean/global_mean if global_mean else np.nan,
            lift10_within_persona=top_mean/mean if mean else np.nan,
            lift_total_top10=top_mean/global_mean if global_mean else np.nan,
            acquisition_rate=rate,global_acquisition_rate=global_rate,top10_acquisition_rate=top_rate,
            rate_lift_persona=rate/global_rate if global_rate else np.nan,
            rate_lift10_within_persona=top_rate/rate if rate else np.nan,
            rate_lift_total_top10=top_rate/global_rate if global_rate else np.nan,
            actual_p10=float(np.quantile(y,.1)),actual_p90=float(np.quantile(y,.9)),
            positive_actual_p10=float(np.quantile(positive,.1)) if len(positive) else np.nan,
            positive_actual_p90=float(np.quantile(positive,.9)) if len(positive) else np.nan,
            weak_evidence=len(g)<MIN_N or len(positive)<MIN_ACTUAL_POSITIVE,
            ranking_informative=bool(np.unique(score).size>1))
        if mean>0 and global_mean>0:
            assert np.isclose(row['lift_persona']*row['lift10_within_persona'],row['lift_total_top10'])
        if rate>0 and global_rate>0:
            assert np.isclose(row['rate_lift_persona']*row['rate_lift10_within_persona'],row['rate_lift_total_top10'])
        lift_rows.append(row)
    lift_status.append(dict(month=m,status='evaluated'))
lift_monthly=pd.DataFrame(lift_rows)
pd.DataFrame(lift_status).to_csv(OUT/'lift_range_status.csv',index=False)
if len(lift_monthly):
    lift_monthly.to_csv(OUT/'persona_lift_total_monthly.csv',index=False)
    aggregate=lift_monthly.groupby('group').agg(
        evaluated_months=('month','nunique'),min_label_n=('n','min'),
        actual_mean_min=('actual_mean','min'),actual_mean_max=('actual_mean','max'),
        acquisition_rate_min=('acquisition_rate','min'),acquisition_rate_max=('acquisition_rate','max'),
        top10_actual_mean_min=('top10_actual_mean','min'),top10_actual_mean_max=('top10_actual_mean','max'),
        top10_acquisition_rate_min=('top10_acquisition_rate','min'),top10_acquisition_rate_max=('top10_acquisition_rate','max'),
        lift_persona_mean=('lift_persona','mean'),lift_persona_min=('lift_persona','min'),lift_persona_max=('lift_persona','max'),
        lift10_within_mean=('lift10_within_persona','mean'),
        lift_total_top10_mean=('lift_total_top10','mean'),lift_total_top10_min=('lift_total_top10','min'),lift_total_top10_max=('lift_total_top10','max'),
        rate_lift_total_top10_mean=('rate_lift_total_top10','mean'),
        strong_months=('weak_evidence',lambda x:int((~x).sum()))).reset_index()
    # Current descriptive range: same context names as backtest; no future actual for May.
    latest=current.loc[current.has_clv].copy()
    latest['group']=np.where(latest.persona_id.notna(),latest.persona_group.astype(str),
        'NoCurrentPersona | '+latest.lifecycle.astype(str))
    pred_range=latest.groupby('group').clv_score.agg(current_predicted_n='size',clv_p10=lambda x:x.quantile(.1),clv_median='median',clv_p90=lambda x:x.quantile(.9)).reset_index()
    allrange=pd.DataFrame([dict(group='ALL_LABELLED_PREDICTIONS',current_predicted_n=np.nan,clv_p10=np.nan,clv_median=np.nan,clv_p90=np.nan)])
    # Aggregate historical labelled cohort differs from current scored cohort; do not merge ALL across populations.
    combined=aggregate.merge(pred_range,on='group',how='outer',validate='one_to_one')
    combined['forecast_month']=RECOMMEND_MONTH;combined['horizon_months']=HORIZON;combined['target']=TARGET
    combined.to_csv(OUT/'persona_lift_and_ranges_summary.csv',index=False)
    print('Điểm acquisition: monthly-first lifts; baseline toàn labelled cohort mỗi tháng')
    display(combined[['group','actual_mean_min','actual_mean_max','clv_p10','clv_median','clv_p90',
        'lift_persona_mean','lift10_within_mean','lift_total_top10_mean','strong_months']].round(4))
    print('Tỷ lệ khách có acquisition: số 0–1, không phải điểm')
    display(combined[['group','acquisition_rate_min','acquisition_rate_max','top10_acquisition_rate_min','top10_acquisition_rate_max','rate_lift_total_top10_mean']].round(4))
    print('Chi tiết từng tháng cuối:')
    display(lift_monthly.loc[lift_monthly.month.eq(EVAL_MONTHS[-1]),['group','n','actual_mean','top10_actual_mean','lift_persona','lift10_within_persona','lift_total_top10','acquisition_rate','top10_acquisition_rate','weak_evidence']].round(4))
else:
    print('Chưa đủ file backtest để tính lift/range thực tế. Xem lift_range_status.csv.')


### Cách đọc bảng mới
Ví dụ một nhóm có lift_persona=3, lift10_within_persona=2 thì top10% nhóm đạt lift_total_top10=6 so toàn cohort, **tại cùng tháng**. Không so hai nhóm chỉ bằng lift nội nhóm.

`actual_mean_min/max` là range của bình quân tháng trong quá khứ; `clv_p10/p90` là range giữa khách của dự báo hiện tại. Không gọi hai range này là khoảng tin cậy. `top10_acquisition_rate` là tỷ lệ có ít nhất một event trong phần top10%, đã tính kỳ vọng khi đồng điểm. Không có nhãn 06/2016 nên không báo actual của source 05/2016.

Hai CSV mới chính: persona_lift_total_monthly.csv và persona_lift_and_ranges_summary.csv. Tất cả average lift là trung bình các lift tính đúng theo tháng. ALL_LABELLED_PREDICTIONS là dòng tổng hợp, không cộng vào tổng nhóm lần nữa.